# camera_105422061350 矩阵的 Plotly 3D 可视化

本 Notebook 展示给定 4x4 齐次变换矩阵所描述的空间关系，并可视化 parent/frame_A 与 child/frame_B 两个坐标系。

## 1) 参数定义

In [ ]:
import numpy as np
import plotly.graph_objects as go

T = np.array([
    [-0.03884323, -0.99906784,  0.01883331,  0.01281536],
    [-0.91665816,  0.02812369, -0.39868146,  0.024272  ],
    [ 0.39778018, -0.03274978, -0.91689605,  0.68936044],
    [ 0.        ,  0.        ,  0.        ,  1.        ],
], dtype=float)

np.set_printoptions(precision=8, suppress=True)
print('输入的 4x4 齐次变换矩阵 T =')
print(T)

## 2) 分解与检查

In [ ]:
R = T[:3, :3]
t = T[:3, 3]

print('平移向量 t =')
print(t)
print('\n旋转矩阵 R =')
print(R)
print('\n4x4 矩阵 T =')
print(T)

# 可选检查：旋转矩阵正交性（R^T R 应约等于 I）
RtR = R.T @ R
I = np.eye(3)
print('\nR^T R =')
print(RtR)
print('\nR^T R - I =')
print(RtR - I)
print('\n正交性近似成立：', np.allclose(RtR, I, atol=1e-6))

## 3) 可视化（Plotly 3D）

In [ ]:
def add_frame(fig, origin, R_mat, frame_name, axis_len=0.12):
    colors = {'x': 'red', 'y': 'green', 'z': 'blue'}
    axes = {'x': R_mat[:, 0], 'y': R_mat[:, 1], 'z': R_mat[:, 2]}

    for axis_name, direction in axes.items():
        end = origin + axis_len * direction
        fig.add_trace(go.Scatter3d(
            x=[origin[0], end[0]],
            y=[origin[1], end[1]],
            z=[origin[2], end[2]],
            mode='lines',
            line=dict(color=colors[axis_name], width=7),
            name=f'{frame_name}-{axis_name}',
            legendgroup=frame_name,
            showlegend=True,
        ))

    fig.add_trace(go.Scatter3d(
        x=[origin[0]],
        y=[origin[1]],
        z=[origin[2]],
        mode='markers+text',
        marker=dict(color='black', size=5),
        text=[frame_name],
        textposition='top center',
        name=f'{frame_name}-origin',
        legendgroup=frame_name,
        showlegend=True,
    ))

axis_len = 0.12
origin_A = np.zeros(3)
R_A = np.eye(3)
origin_B = t
R_B = R

fig = go.Figure()
add_frame(fig, origin_A, R_A, 'parent/frame_A', axis_len=axis_len)
add_frame(fig, origin_B, R_B, 'child/frame_B', axis_len=axis_len)

# 两个原点连线（黑色虚线）
fig.add_trace(go.Scatter3d(
    x=[origin_A[0], origin_B[0]],
    y=[origin_A[1], origin_B[1]],
    z=[origin_A[2], origin_B[2]],
    mode='lines',
    line=dict(color='black', dash='dash', width=5),
    name='parent-origin -> child-origin',
    showlegend=True,
))

# 尽量等比例显示
points = [origin_A, origin_B]
for i in range(3):
    points.append(origin_A + axis_len * R_A[:, i])
    points.append(origin_B + axis_len * R_B[:, i])
all_points = np.vstack(points)
mins = all_points.min(axis=0)
maxs = all_points.max(axis=0)
center = (mins + maxs) / 2.0
span = float(np.max(maxs - mins))
if span < 1e-9:
    span = 1.0
half = span * 0.65
x_range = [center[0] - half, center[0] + half]
y_range = [center[1] - half, center[1] + half]
z_range = [center[2] - half, center[2] + half]

fig.update_layout(
    title='camera_105422061350: 4x4 齐次变换矩阵 3D 可视化',
    scene=dict(
        xaxis=dict(title='X', range=x_range),
        yaxis=dict(title='Y', range=y_range),
        zaxis=dict(title='Z', range=z_range),
        aspectmode='cube',
    ),
    legend=dict(itemsizing='constant'),
    margin=dict(l=0, r=0, b=0, t=45),
)

fig.show()

## 4) 结论说明

该矩阵 T 表示 **child/frame_B 相对 parent/frame_A 的位姿**：
- 旋转矩阵 R 描述 child 坐标轴相对于 parent 的方向关系；
- 平移向量 t 描述 child 原点在 parent 坐标系下的位置；
- 图中的黑色虚线连接了两个坐标系原点，直观反映了这段平移。